# 📊 Analisis Saham Multi-Agent AI
### Powered by Mistral AI × yfinance

Notebook ini menganalisis saham pilihan kamu menggunakan **5 agen AI**:

| Agen | Tugasnya |
|------|----------|
| 🔬 **Analis Fundamental** | Apakah perusahaannya sehat? Mahal atau murah? |
| 📰 **Pembaca Berita** | Berita apa yang beredar? Positif atau negatif? |
| 📈 **Analis Teknikal** | Tren harga ke mana? Kapan waktu yang bagus? |
| 🎯 **Penyusun Strategi** | Beli di harga berapa? Target & stop loss berapa? |
| ⚖️ **Komite Keputusan** | Kesimpulan akhir: Beli, Tahan, atau Jual? |

---
**Cara pakai:**
1. Jalankan sel install (sekali saja)
2. Isi API key Mistral (atau sudah otomatis dari .env)
3. Ganti kode saham di sel INPUT
4. Klik **Run All** (▶▶) — tunggu ±5 menit

> ⚠️ **Disclaimer:** Hasil analisis ini adalah pendapat AI, BUKAN saran investasi resmi.

In [ ]:
# ─── Install library (jalankan sekali) ───────────────────────────────────────
!pip install -q yfinance langchain-mistralai langchain-core pandas matplotlib python-dotenv ddgs

In [ ]:
import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
load_dotenv()

MISTRAI_API_KEY = os.getenv('MISTRAL_API_KEY', '')
if not MISTRAI_API_KEY:
    MISTRAI_API_KEY = input('Masukkan Mistral API Key kamu: ').strip()

def tanya_ai(prompt: str, model='mistral-large-latest') -> str:
    try:
        llm = ChatMistralAI(model=model, api_key=MISTRAI_API_KEY, temperature=0.3)
        resp = llm.invoke([HumanMessage(content=prompt)])
        return resp.content
    except Exception as e:
        return f'[Error AI: {e}]'

print('✅ Setup selesai!')

In [ ]:
# ════════════════════════════════════════════════════════
# 🎯 INPUT — Ganti kode saham di sini
# ════════════════════════════════════════════════════════

# Saham Indonesia: tambahkan .JK  →  TLKM.JK  BBCA.JK  GOTO.JK  ASII.JK
# Saham US       : langsung saja  →  AAPL     NVDA     TSLA

TICKER = 'TLKM.JK'           # ← GANTI DI SINI
NAMA_SAHAM = 'Telkom Indonesia'  # ← nama lengkap

print(f'🎯 Saham yang akan dianalisis: {TICKER} ({NAMA_SAHAM})')

In [ ]:
# ─── Ambil data dari Yahoo Finance ───────────────────────────────────────────
print(f'⏳ Mengambil data {TICKER}...')

tk   = yf.Ticker(TICKER)
info = tk.info
hist = tk.history(period='1y')
if hist.empty:
    hist = tk.history(period='6mo')

try:
    fin = tk.financials
    bal = tk.balance_sheet
    cf  = tk.cashflow
except:
    fin, bal, cf = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

harga     = info.get('currentPrice') or info.get('regularMarketPrice') or float(hist['Close'].iloc[-1])
nama      = info.get('longName', NAMA_SAHAM)
sektor    = info.get('sector', 'Tidak diketahui')
pe        = info.get('trailingPE') or info.get('forwardPE')
pb        = info.get('priceToBook')
div_yield = info.get('dividendYield', 0)
mcap      = info.get('marketCap', 0)
beta      = info.get('beta')
w52h      = info.get('fiftyTwoWeekHigh')
w52l      = info.get('fiftyTwoWeekLow')

def fmt_rp(n):
    if not n: return 'N/A'
    if n >= 1e12: return f'Rp {n/1e12:.1f} T'
    if n >= 1e9:  return f'Rp {n/1e9:.1f} M'
    return f'{n:,.0f}'

print(f"\n{'='*52}\n  {nama} ({TICKER})\n{'='*52}")
print(f'  Harga        : {harga:,.0f}')
print(f'  Sektor       : {sektor}')
print(f'  Market Cap   : {fmt_rp(mcap)}')
print(f'  P/E Ratio    : {f"{pe:.1f}x" if pe else "N/A"}')
print(f'  P/B Ratio    : {f"{pb:.1f}x" if pb else "N/A"}')
print(f'  Dividen      : {f"{div_yield*100:.1f}%" if div_yield else "Tidak ada"}')
print(f'  52w High/Low : {f"{w52h:,.0f}" if w52h else "N/A"} / {f"{w52l:,.0f}" if w52l else "N/A"}')
print(f'  Beta         : {f"{beta:.2f}" if beta else "N/A"}')
print(f"{'='*52}")

In [ ]:
# ─── Grafik Harga + RSI + MACD ───────────────────────────────────────────────
close  = hist['Close']
volume = hist['Volume']

# Indikator
ma20   = close.rolling(20).mean()
ma50   = close.rolling(50).mean()
ma200  = close.rolling(200).mean()

delta  = close.diff()
avg_g  = delta.clip(lower=0).rolling(14).mean()
avg_l  = (-delta).clip(lower=0).rolling(14).mean()
rs     = avg_g / avg_l.replace(0, np.nan)
rsi    = 100 - (100 / (1 + rs))

ema12  = close.ewm(span=12).mean()
ema26  = close.ewm(span=26).mean()
macd   = ema12 - ema26
signal = macd.ewm(span=9).mean()
hist_m = macd - signal

bb_mid = close.rolling(20).mean()
bb_std = close.rolling(20).std()
bb_up  = bb_mid + 2*bb_std
bb_lo  = bb_mid - 2*bb_std

fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 1, 1], hspace=0.07)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#8b949e', labelsize=8)
    for sp in ax.spines.values(): sp.set_color('#30363d')
    ax.grid(color='#21262d', linewidth=0.4)

ax1.fill_between(close.index, bb_up, bb_lo, alpha=0.08, color='#58a6ff')
ax1.plot(close.index, close,  color='#e6edf3', lw=1.5, label='Harga Penutupan')
ax1.plot(close.index, ma20,   color='#f0883e', lw=1,   label='MA 20 (jangka pendek)')
ax1.plot(close.index, ma50,   color='#3fb950', lw=1,   label='MA 50 (jangka menengah)')
if not ma200.dropna().empty:
    ax1.plot(close.index, ma200, color='#bc8cff', lw=0.8, ls='--', label='MA 200 (jangka panjang)')

now_price = float(close.iloc[-1])
ax1.axhline(now_price, color='#ff7b72', lw=0.8, ls=':')
ax1.text(close.index[-1], now_price, f'  {now_price:,.0f}', color='#ff7b72', va='center', fontsize=8)
ax1.set_title(f'{nama} ({TICKER}) — Analisis Teknikal', color='#e6edf3', fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('Harga', color='#8b949e', fontsize=8)
ax1.legend(loc='upper left', facecolor='#161b22', edgecolor='#30363d', labelcolor='#8b949e', fontsize=7)

rsi_now = float(rsi.iloc[-1])
ax2.plot(rsi.index, rsi, color='#a5d6ff', lw=1.2)
ax2.axhline(70, color='#ff7b72', lw=0.7, ls='--')
ax2.axhline(30, color='#3fb950', lw=0.7, ls='--')
ax2.fill_between(rsi.index, rsi, 70, where=(rsi >= 70), alpha=0.25, color='#ff7b72')
ax2.fill_between(rsi.index, rsi, 30, where=(rsi <= 30), alpha=0.25, color='#3fb950')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI', color='#8b949e', fontsize=8)
rc = '#ff7b72' if rsi_now > 70 else ('#3fb950' if rsi_now < 30 else '#a5d6ff')
ax2.text(rsi.index[-1], rsi_now, f'  {rsi_now:.0f}', color=rc, va='center', fontsize=8)

hist_colors = ['#3fb950' if v >= 0 else '#ff7b72' for v in hist_m]
ax3.bar(hist_m.index, hist_m, color=hist_colors, alpha=0.6, width=1)
ax3.plot(macd.index, macd,   color='#58a6ff', lw=1, label='MACD')
ax3.plot(signal.index, signal, color='#f0883e', lw=1, label='Signal')
ax3.axhline(0, color='#30363d', lw=0.5)
ax3.set_ylabel('MACD', color='#8b949e', fontsize=8)
ax3.legend(loc='upper left', facecolor='#161b22', edgecolor='#30363d', labelcolor='#8b949e', fontsize=7)

plt.setp(ax1.get_xticklabels(), visible=False)
plt.setp(ax2.get_xticklabels(), visible=False)
plt.tight_layout()
plt.show()

# Simpan nilai indikator untuk agen
_rsi     = round(float(rsi.iloc[-1]), 1)
_macd_v  = round(float(macd.iloc[-1]), 4)
_signal_v= round(float(signal.iloc[-1]), 4)
_ma20_v  = round(float(ma20.iloc[-1]), 0)
_ma50_v  = round(float(ma50.iloc[-1]), 0)
_bb_up_v = round(float(bb_up.iloc[-1]), 0)
_bb_lo_v = round(float(bb_lo.iloc[-1]), 0)
_gc      = (_ma20_v > _ma50_v)  # Golden Cross?
_d7      = round((close.iloc[-1]-close.iloc[-7])/close.iloc[-7]*100, 2) if len(close)>=7 else 0
_d30     = round((close.iloc[-1]-close.iloc[-30])/close.iloc[-30]*100, 2) if len(close)>=30 else 0
_d90     = round((close.iloc[-1]-close.iloc[-90])/close.iloc[-90]*100, 2) if len(close)>=90 else 0

print('\n📊 Indikator Teknikal Hari Ini:')
print(f'  RSI  : {_rsi} → {"⚠️ Overbought (terlalu banyak yg beli, biasanya akan koreksi)" if _rsi>70 else ("✅ Oversold (terlalu banyak yg jual, potensi naik)" if _rsi<30 else "✅ Normal")}')
print(f'  MACD : {"🟢 Bullish (momentum naik)" if _macd_v > _signal_v else "🔴 Bearish (momentum turun)"}')
print(f'  Tren : {"🟢 Golden Cross (MA20>MA50) — sinyal positif" if _gc else "🔴 Death Cross (MA20<MA50) — sinyal negatif"}')
print(f'  Perubahan: 7h={_d7:+.1f}% | 30h={_d30:+.1f}% | 90h={_d90:+.1f}%')

In [ ]:
# ════════════════════════════════════════════════════════
# AGEN 1 — 🔬 Analis Fundamental
# ════════════════════════════════════════════════════════
print('🔬 Agen 1: Analis Fundamental sedang bekerja...')

def _keuangan(df, rows, label):
    if df.empty: return ''
    lines = [f'\n{label}:']
    cols = list(df.columns[:4])
    for r in rows:
        if r in df.index:
            vs = [f'{v/1e9:.1f}M' if pd.notna(v) and abs(v)>=1e9 else f'{v:,.0f}' 
                  for v in df.loc[r, cols] if pd.notna(v)]
            lines.append(f'  {r}: {" | ".join(vs)}')
    return '\n'.join(lines)

data_fin = f"""
SAHAM: {TICKER} — {nama}\nSektor: {sektor}\nHarga: {harga:,.0f}\nMarket Cap: {fmt_rp(mcap)}
P/E: {f"{pe:.1f}x" if pe else 'N/A'} | P/B: {f"{pb:.1f}x" if pb else 'N/A'}
Dividen: {f"{div_yield*100:.1f}%" if div_yield else 'Tidak ada'} | Beta: {f"{beta:.2f}" if beta else 'N/A'}
52w: High {f"{w52h:,.0f}" if w52h else 'N/A'} / Low {f"{w52l:,.0f}" if w52l else 'N/A'}
Target analis: {info.get('targetMeanPrice','N/A')} | Rekomendasi: {info.get('recommendationKey','N/A')}
"""
data_fin += _keuangan(fin, ['Total Revenue','Gross Profit','Operating Income','Net Income'], 'Laba Rugi (Miliar)')
data_fin += _keuangan(bal, ['Total Assets','Total Liabilities Net Minority Interest','Cash And Cash Equivalents'], 'Neraca')
data_fin += _keuangan(cf,  ['Free Cash Flow','Operating Cash Flow'], 'Arus Kas')

prompt1 = f"""Kamu adalah analis saham yang menjelaskan kondisi saham kepada orang awam yang belum pernah investasi.

DATA:\n{data_fin}

TUGAS — Buat analisis fundamental dengan bahasa sederhana:
- Setiap istilah teknikal WAJIB dijelaskan dalam tanda kurung
- Gunakan analogi sehari-hari jika perlu
- Bandingkan angka dengan patokan umum (P/E normal 10-20x, dst)
- Gunakan Bahasa Indonesia santai, seperti bicara ke teman

Format:
## 🔬 Analisis Fundamental — {nama}

### 🏥 Apakah Perusahaannya Sehat?
[analisis sederhana kondisi keuangan]

### 💰 Harganya Mahal atau Murah?
[penjelasan valuasi dengan analogi]

### 📈 Apakah Perusahaan Tumbuh?
[tren revenue dan profit dalam bahasa sederhana]

### 💸 Dividen — Dapat Bagian Untung?
[apakah ada dividen dan berapa]

### ✅ Kesimpulan
[1-2 kalimat sederhana: layak atau tidak dari sisi fundamental]"""

hasil_f = tanya_ai(prompt1)
display(Markdown(hasil_f))

In [ ]:
# ════════════════════════════════════════════════════════
# AGEN 2 — 📰 Pembaca Berita & Sentimen
# ════════════════════════════════════════════════════════
print('📰 Agen 2: Sedang membaca berita terkini...')

berita = []
try:
    from ddgs import DDGS
    q = f"{TICKER.replace('.JK','').replace('.','')} {NAMA_SAHAM} saham berita"
    for r in list(DDGS().news(q, max_results=15)):
        berita.append(f"- [{r.get('date','')}] {r.get('title','')} — {r.get('body','')[:120]}")
    print(f'  ✅ {len(berita)} berita ditemukan')
except Exception as e:
    print(f'  ⚠️ {e}')

try:
    for n in (tk.news or [])[:5]:
        t = n.get('content', {})
        title = t.get('title', n.get('title', ''))
        if title: berita.append(f'- [yfinance] {title}')
except: pass

berita_txt = '\n'.join(berita[:20]) if berita else 'Tidak ada berita'

prompt2 = f"""Kamu adalah analis berita saham yang menjelaskan situasi kepada investor pemula.

BERITA TERBARU {TICKER} — {nama}:\n{berita_txt}

Buat ringkasan yang mudah dipahami:
- Pisahkan berita POSITIF (bagus) dan NEGATIF (buruk untuk saham)
- Jelaskan KENAPA setiap berita itu penting untuk investor
- Berikan skor sentimen keseluruhan
- Bahasa santai dan mudah dipahami

Format:
## 📰 Analisis Berita — {nama}

### 😊 Berita Positif (Bagus untuk Saham)
[daftar + penjelasan kenapa bagus]

### 😟 Berita Negatif (Perlu Diwaspadai)
[daftar + penjelasan risikonya]

### 🌡️ Suhu Pasar
**Skor Sentimen:** [Sangat Positif / Positif / Netral / Negatif / Sangat Negatif]
[penjelasan 2-3 kalimat]

### ⚠️ Yang Perlu Diawasi
[2-3 hal penting untuk investor pemula]"""

hasil_b = tanya_ai(prompt2)
display(Markdown(hasil_b))

In [ ]:
# ════════════════════════════════════════════════════════
# AGEN 3 — 📈 Analis Teknikal
# ════════════════════════════════════════════════════════
print('📈 Agen 3: Membaca grafik dan indikator teknikal...')

vol_avg = float(volume.rolling(20).mean().iloc[-1])
vol_now = int(volume.iloc[-1])

data_tek = f"""
SAHAM: {TICKER} | Harga: {harga:,.0f}
Perubahan: 7h={_d7:+.1f}% | 30h={_d30:+.1f}% | 90h={_d90:+.1f}%

INDIKATOR:
RSI(14)={_rsi} [overbought>70, oversold<30]
MACD={_macd_v} vs Signal={_signal_v} → {'MACD di ATAS signal (momentum naik)' if _macd_v>_signal_v else 'MACD di BAWAH signal (momentum turun)'}
MA20={_ma20_v:,.0f} | MA50={_ma50_v:,.0f} | Harga {'DI ATAS' if harga>_ma20_v else 'DI BAWAH'} MA20
Golden/Death Cross: {'Golden Cross (MA20>MA50) = sinyal beli' if _gc else 'Death Cross (MA20<MA50) = sinyal jual'}
Bollinger: Upper={_bb_up_v:,.0f} | Lower={_bb_lo_v:,.0f} | Harga {'dekat batas atas (mahal secara teknikal)' if harga>_bb_up_v*0.97 else ('dekat batas bawah (murah secara teknikal)' if harga<_bb_lo_v*1.03 else 'di tengah (normal)')}
Volume: {vol_now:,} vs rata-rata {vol_avg:,.0f} = {vol_now/vol_avg:.1f}x ({'volume tinggi = ada pergerakan besar' if vol_now/vol_avg>1.5 else 'volume normal'})
52w: High={f"{w52h:,.0f}" if w52h else 'N/A'} | Low={f"{w52l:,.0f}" if w52l else 'N/A'}
"""

prompt3 = f"""Kamu adalah analis teknikal saham yang menjelaskan grafik kepada investor pemula.

DATA TEKNIKAL:\n{data_tek}

Jelaskan kondisi teknikal dalam bahasa yang sangat mudah dipahami:
- WAJIB jelaskan setiap indikator dalam bahasa sehari-hari dengan analogi
  Contoh RSI: 'RSI 75 = banyak orang buru-buru beli dalam waktu singkat, biasanya akan koreksi dulu'
  Contoh MA: 'MA20 = rata-rata harga 20 hari. Kalau harga di atasnya, trennya bagus'
- Sebutkan level support (harga lantai) dan resistance (harga atap) yang penting
- Berikan kesimpulan: apakah sekarang momen bagus untuk beli atau lebih baik tunggu

Format:
## 📈 Analisis Teknikal — {nama}

### 🌊 Arah Tren
[tren naik/turun/sideways dengan penjelasan sederhana]

### 🔍 Penjelasan Indikator (Bahasa Awam)
[RSI, MACD, MA, Bollinger — masing-masing 2-3 kalimat sederhana]

### 🏠 Harga Penting
**Support (lantai harga):** [harga] — [kenapa ini penting]
**Resistance (atap harga):** [harga] — [kenapa ini penting]

### ⏰ Momen Beli: Sekarang atau Tunggu?
[kesimpulan sederhana dan konkret]"""

hasil_t = tanya_ai(prompt3)
display(Markdown(hasil_t))

In [ ]:
# ════════════════════════════════════════════════════════
# AGEN 4 — 🎯 Penyusun Strategi
# ════════════════════════════════════════════════════════
print('🎯 Agen 4: Menyusun strategi trading...')

ringkasan = f"""
FUNDAMENTAL (ringkasan): {hasil_f[:500]}
BERITA (ringkasan): {hasil_b[:400]}
TEKNIKAL: RSI={_rsi} | MACD={'Bullish' if _macd_v>_signal_v else 'Bearish'} | Tren={'Golden Cross' if _gc else 'Death Cross'}
Harga={harga:,.0f} | BB Lower={_bb_lo_v:,.0f} | BB Upper={_bb_up_v:,.0f} | Perubahan 30h={_d30:+.1f}%
"""

prompt4 = f"""Kamu adalah penasihat trading yang memberikan strategi konkret kepada investor pemula.

DATA ANALISIS:\n{ringkasan}\nSAHAM: {TICKER} | Harga sekarang: {harga:,.0f}

Buat strategi yang spesifik, praktis, dan mudah dipahami:
- Berikan ANGKA SPESIFIK (bukan range terlalu lebar)
- Jelaskan alasan setiap keputusan dengan bahasa sederhana
- Wajib jelaskan stop loss dalam bahasa awam
- Sertakan strategi cicil (DCA) jika cocok

Format:
## 🎯 Strategi Trading — {nama}

### 🛒 Harga Beli (Entry)
**Beli di:** [harga spesifik atau range kecil]
[alasan dalam bahasa sederhana]

### 🎁 Target Keuntungan
**Target 1:** [harga] = potensi untung +X%
**Target 2:** [harga] = potensi untung +X%
[kapan dan kenapa keluar di level ini]

### 🛡️ Batas Kerugian (Stop Loss)
**Stop Loss di:** [harga] = maksimal rugi X%
**Artinya:** Kalau harga turun sampai [harga], langsung jual untuk batasi kerugian
[penjelasan kenapa level ini dipilih]

### 📊 Untung vs Rugi
**Maksimal rugi:** X% | **Potensi untung:** Y% | **Rasio:** 1:Z
[apakah rasio ini bagus? Rasio minimal 1:2 dianggap layak]

### 📅 Berapa Lama?
**Horizon:** [durasi] — [kapan rencana keluar]

### 💼 Alokasi Modal
[berapa % dari total modal yang disarankan]

### 🔄 Strategi Cicil (DCA)
[rencana cicil masuk jika cocok]"""

hasil_s = tanya_ai(prompt4)
display(Markdown(hasil_s))

In [ ]:
# ════════════════════════════════════════════════════════
# AGEN 5 — ⚖️ Komite Keputusan Final
# ════════════════════════════════════════════════════════
print('⚖️ Agen 5: Komite keputusan bersidang...')

semua = f"""
SAHAM: {TICKER} — {nama} | Harga: {harga:,.0f} | Sektor: {sektor}

FUNDAMENTAL: {hasil_f[:600]}
BERITA: {hasil_b[:500]}
TEKNIKAL: {hasil_t[:500]}
STRATEGI: {hasil_s[:500]}
"""

prompt5 = f"""Kamu adalah ketua komite investasi yang membuat keputusan final berdasarkan semua analisis tim.
Bersikaplah JUJUR dan OBJEKTIF — sampaikan kondisi apa adanya.

SEMUA ANALISIS:\n{semua}

Buat laporan keputusan final yang mudah dipahami investor pemula:
- Verdikt JELAS: BELI / TAHAN / JUAL
- Alasan dalam bahasa sangat sederhana
- Skenario terbaik dan terburuk
- Skor keyakinan 1-10
- Peringatan risiko untuk pemula

Format:
## ⚖️ Keputusan Final — {nama}

---

# [🟢 BELI / 🟡 TAHAN / 🔴 JUAL]
**Skor Keyakinan: X/10**

---

### 📝 Ringkasan (3 Kalimat)
[kondisi saham dalam 3 kalimat yang langsung mudah dipahami]

### ✅ Alasan Utama Keputusan
1. [alasan 1 — konkret dan sederhana]
2. [alasan 2]
3. [alasan 3]

### 🌤️ Kalau Semua Berjalan Lancar
[skenario terbaik yang bisa terjadi]

### ⛈️ Kalau Ada yang Salah
[skenario terburuk — jujur soal risiko]

### ⚠️ 3 Risiko Utama yang Harus Kamu Tahu
1. [risiko 1]
2. [risiko 2]
3. [risiko 3]

### 💡 Pesan untuk Investor Pemula
[1-2 kalimat saran realistis tentang investasi saham]"""

hasil_v = tanya_ai(prompt5)
display(Markdown(hasil_v))

In [ ]:
# ════════════════════════════════════════════════════════
# DASHBOARD RINGKASAN FINAL
# ════════════════════════════════════════════════════════
verdikt = '🟢 BELI' if 'BELI' in hasil_v.upper() else ('🔴 JUAL' if 'JUAL' in hasil_v.upper() else '🟡 TAHAN')

pe_ket  = ('Mahal (>25x)' if pe and pe>25 else ('Murah (<10x)' if pe and pe<10 else 'Wajar')) if pe else 'N/A'
beta_ket= ('Sangat volatile (>1.5)' if beta and beta>1.5 else ('Volatile (>1)' if beta and beta>1 else 'Stabil')) if beta else 'N/A'
rsi_ket = '⚠️ Overbought' if _rsi>70 else ('✅ Oversold' if _rsi<30 else '✅ Normal')
macd_ket= '🟢 Bullish' if _macd_v>_signal_v else '🔴 Bearish'
tren_ket= '🟢 Golden Cross' if _gc else '🔴 Death Cross'

dash = f"""
# 📊 Dashboard — {nama} ({TICKER})
*Analisis AI: {datetime.now().strftime("%d %B %Y, %H:%M")}*

---

## {verdikt}

| Metrik | Nilai | Keterangan |
|--------|-------|------------|
| 💰 Harga | **{harga:,.0f}** | — |
| 📈 Perubahan 30 Hari | **{_d30:+.1f}%** | {'↑ Naik' if _d30>=0 else '↓ Turun'} |
| 📊 P/E Ratio | **{f"{pe:.1f}x" if pe else 'N/A'}** | {pe_ket} |
| 💸 Dividen | **{f"{div_yield*100:.1f}%" if div_yield else 'Tidak ada'}** | — |
| 🎢 Beta (Volatilitas) | **{f"{beta:.2f}" if beta else 'N/A'}** | {beta_ket} |
| 🔍 RSI | **{_rsi}** | {rsi_ket} |
| 📉 MACD | — | {macd_ket} |
| 📐 Tren MA | — | {tren_ket} |
| 🏦 Market Cap | **{fmt_rp(mcap)}** | — |
| 📅 52w High | **{f"{w52h:,.0f}" if w52h else 'N/A'}** | {'⚠️ Dekat puncak' if w52h and harga>=0.95*float(w52h) else '—'} |
| 📅 52w Low | **{f"{w52l:,.0f}" if w52l else 'N/A'}** | {'✅ Dekat dasar' if w52l and harga<=1.05*float(w52l) else '—'} |

---

## ✅ 5 Agen AI Selesai Bertugas

| Agen | Status |
|------|--------|
| 🔬 Analis Fundamental | ✅ Selesai |
| 📰 Pembaca Berita | ✅ Selesai |
| 📈 Analis Teknikal | ✅ Selesai |
| 🎯 Penyusun Strategi | ✅ Selesai |
| ⚖️ Komite Keputusan | ✅ Selesai |

---
> ⚠️ **Disclaimer:** Ini adalah pendapat AI, BUKAN saran investasi resmi.
> Selalu lakukan riset sendiri sebelum berinvestasi. Nilai saham bisa naik maupun turun.
"""

display(Markdown(dash))

# Simpan laporan
lap_path = Path(f"Analisis_{TICKER.replace('.','_')}_{datetime.now().strftime('%Y%m%d_%H%M')}.md")
lap_path.write_text(f"""# Laporan Analisis: {nama} ({TICKER})\n*{datetime.now().strftime('%d %B %Y %H:%M')}*\n\n{hasil_f}\n\n---\n\n{hasil_b}\n\n---\n\n{hasil_t}\n\n---\n\n{hasil_s}\n\n---\n\n{hasil_v}\n""", encoding='utf-8')
print(f"\n💾 Laporan disimpan: {lap_path.absolute()}")
print(f"\n🎉 Selesai! Verdikt: {verdikt}")